In [3]:
import pandas as pd

matches = pd.read_csv("../data/raw/WorldCupMatches.csv")

print("Original shape:", matches.shape)
# matches.head()

matches.isnull().sum()
matches_clean = matches.dropna(how="all")

print("Shape after removing fully empty rows:", matches_clean.shape)

required_columns = [
    "Year",
    "Datetime",
    "Stage",
    "Home Team Name",
    "Away Team Name",
    "Home Team Goals",
    "Away Team Goals"
]

matches_clean = matches_clean.dropna(subset=required_columns)

print("Shape after removing invalid match rows:", matches_clean.shape)

Original shape: (4572, 20)
Shape after removing fully empty rows: (852, 20)
Shape after removing invalid match rows: (852, 20)


In [4]:
duplicates_count = matches_clean.duplicated().sum()
print("Duplicate rows:", duplicates_count)

matches_clean = matches_clean.drop_duplicates()
print("Shape after removing duplicates:", matches_clean.shape)

Duplicate rows: 16
Shape after removing duplicates: (836, 20)


In [5]:
# missing_values = matches_clean.isnull().sum()

# print("Missing values after removing empty rows and duplicates:")
# print(missing_values[missing_values > 0])

matches_clean["Attendance"] = matches_clean["Attendance"].fillna(
    matches_clean["Attendance"].median()
)

print("Missing values after handling Attendance:")
print(matches_clean.isnull().sum()[matches_clean.isnull().sum() > 0])

Missing values after handling Attendance:
Series([], dtype: int64)


In [6]:
matches_clean["Datetime"] = pd.to_datetime(
    matches_clean["Datetime"],
    errors="coerce"
)

print(matches_clean["Datetime"].dtype)
print(matches_clean["Datetime"].isnull().sum())

invalid_dates = matches_clean["Datetime"].isnull().sum()
print("Invalid dates:", invalid_dates)

datetime64[ns]
10
Invalid dates: 10


In [7]:
home_teams = matches_clean["Home Team Name"].sort_values().unique()
away_teams = matches_clean["Away Team Name"].sort_values().unique()

# print("Home teams:")
# print(home_teams)

# print("Away teams:")
# print(away_teams)

all_teams = pd.concat([
    matches_clean["Home Team Name"],
    matches_clean["Away Team Name"]
]).sort_values().unique()

all_teams

array(['Algeria', 'Angola', 'Argentina', 'Australia', 'Austria',
       'Belgium', 'Bolivia', 'Brazil', 'Bulgaria', 'Cameroon', 'Canada',
       'Chile', 'China PR', 'Colombia', 'Costa Rica', 'Croatia', 'Cuba',
       'Czech Republic', 'Czechoslovakia', "C�te d'Ivoire", 'Denmark',
       'Dutch East Indies', 'Ecuador', 'Egypt', 'El Salvador', 'England',
       'France', 'German DR', 'Germany', 'Germany FR', 'Ghana', 'Greece',
       'Haiti', 'Honduras', 'Hungary', 'IR Iran', 'Iran', 'Iraq',
       'Israel', 'Italy', 'Jamaica', 'Japan', 'Korea DPR',
       'Korea Republic', 'Kuwait', 'Mexico', 'Morocco', 'Netherlands',
       'New Zealand', 'Nigeria', 'Northern Ireland', 'Norway', 'Paraguay',
       'Peru', 'Poland', 'Portugal', 'Romania', 'Russia', 'Saudi Arabia',
       'Scotland', 'Senegal', 'Serbia', 'Slovakia', 'Slovenia',
       'South Africa', 'Soviet Union', 'Spain', 'Sweden', 'Switzerland',
       'Togo', 'Tunisia', 'Turkey', 'USA', 'Ukraine', 'Uruguay', 'Wales',
       'Yugosl

In [9]:
team_name_mapping = {
    "Germany FR": "Germany",
    "rn\">Germany": "Germany",
    "rn\">Republic of Ireland": "Republic of Ireland",
    "rn\">Serbia and Montenegro": "Serbia and Montenegro",
    "rn\">Trinidad and Tobago": "Trinidad and Tobago",
    "Soviet Union": "Russia",
    "Côte d'Ivoire": "Ivory Coast",
    "IR Iran": "Iran",
    "USA": "United States"
}

matches_clean["Home Team Name"] = matches_clean["Home Team Name"].replace(team_name_mapping)
matches_clean["Away Team Name"] = matches_clean["Away Team Name"].replace(team_name_mapping)

all_teams_cleaned = pd.concat([
    matches_clean["Home Team Name"],
    matches_clean["Away Team Name"]
]).sort_values().unique()

all_teams_cleaned

array(['Algeria', 'Angola', 'Argentina', 'Australia', 'Austria',
       'Belgium', 'Bolivia', 'Brazil', 'Bulgaria', 'Cameroon', 'Canada',
       'Chile', 'China PR', 'Colombia', 'Costa Rica', 'Croatia', 'Cuba',
       'Czech Republic', 'Czechoslovakia', "C�te d'Ivoire", 'Denmark',
       'Dutch East Indies', 'Ecuador', 'Egypt', 'El Salvador', 'England',
       'France', 'German DR', 'Germany', 'Ghana', 'Greece', 'Haiti',
       'Honduras', 'Hungary', 'Iran', 'Iraq', 'Israel', 'Italy',
       'Jamaica', 'Japan', 'Korea DPR', 'Korea Republic', 'Kuwait',
       'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Nigeria',
       'Northern Ireland', 'Norway', 'Paraguay', 'Peru', 'Poland',
       'Portugal', 'Republic of Ireland', 'Romania', 'Russia',
       'Saudi Arabia', 'Scotland', 'Senegal', 'Serbia',
       'Serbia and Montenegro', 'Slovakia', 'Slovenia', 'South Africa',
       'Spain', 'Sweden', 'Switzerland', 'Togo', 'Trinidad and Tobago',
       'Tunisia', 'Turkey', 'Ukraine', 'Uni

## Duplicate Rows
The dataset was checked for fully duplicated rows using `.duplicated()`.
Duplicated rows were removed with `.drop_duplicates()` to avoid counting the same match more than once.

Duplicate rows: 16
Shape after removing duplicates: 836, 20 (rows, columns) 

## Missing Values
After removing empty and invalid rows, the remaining missing values were reviewed by column.
The `Attendance` column was filled using the median instead of zero because a missing attendance value does not mean that no people attended the match. The median is more robust than the mean when the data contains very large stadium attendance values.

## Date Conversion
The `Datetime` column was converted from text to a proper datetime type using `pd.to_datetime()`.
Invalid date values were converted to `NaT` using `errors="coerce"` so they can be detected and handled safely.

## Team Name Standardization
Team names were standardized to avoid treating the same national team as different entities.
For example, `Germany FR` and `rn">Germany` were mapped to `Germany`, while `USA` was mapped to `United States`.
This step is important because future aggregations by team depend on consistent team names.

In [10]:
matches_clean.to_csv("../data/processed/clean_partial.csv", index=False)

print("Saved file: ../data/processed/clean_partial.csv")
print("Final shape:", matches_clean.shape)

Saved file: ../data/processed/clean_partial.csv
Final shape: (836, 20)
